In [ ]:
import matplotlib

matplotlib.rcParams['pdf.fonttype'] = 42  # to edit text in Illustrator
import pandas as pd


%load_ext autoreload
%autoreload 2
from seismosocialdistancing_simple import SeismoNoise

In [ ]:
NETWORK = "XG"
STATION = "BB01"
LOCATION = ""
CHANNEL = "HHZ"
dataset = "Skience2026"
time_zone = "Europe/Zurich"
sitedesc = "on ice"


## 10. Reload a saved PPSD from disk

Previously computed PPSDs can be reloaded from `.npz` files using
`PPSD.load_npz()`. This is much faster than recomputing from raw waveforms —
especially useful when you've accumulated weeks or months of data.

In [ ]:
from obspy.signal.spectral_estimation import PPSD

ppsd = PPSD.load_npz(f"./{NETWORK}.{STATION}.{LOCATION}.{CHANNEL}.npz")

print(f"Loaded PPSD — {len(ppsd.current_times_used)} segment(s)")
print(f"  Station : {ppsd.id}")
print(f"  Period  : {ppsd.period_bin_centers[0]:.3f} – {ppsd.period_bin_centers[-1]:.1f} s")
print(f"  Time    : {ppsd.current_times_used[0].date} → {ppsd.current_times_used[-1].date}")

In [ ]:
# Let's check we got the right stuff
target_period = 0.1

ppsd.plot(max_percentage=10)
ppsd.plot_temporal(target_period)
ppsd.plot_spectrogram(clim=(-160,-80))

In [ ]:
freqs = [(0.1,1.0),(1.0,20.0),(4.0,14.0),(4.0,20.0),(2.0,100.0)]
ind_times = pd.DatetimeIndex([d.datetime for d in ppsd.current_times_used])
data = pd.DataFrame(ppsd.psd_values, index=ind_times, columns=1./ppsd.period_bin_centers)
data = data.sort_index(axis=1)
displacement_RMS = {}
displacement_RMS[f"{NETWORK}.{STATION}.{LOCATION}.{CHANNEL}"] = seismosocialdistancing.df_rms(data, freqs, output="DISP")

In [ ]:
displacement_RMS[f"{NETWORK}.{STATION}.{LOCATION}.{CHANNEL}"].head()

In [ ]:
args = {'band':"4.0-14.0",       # might be None or commented ("4.0-14.0" per default) or any of the tupples in freqs
        'time_zone':time_zone,   # required for clockplots
        'sitedesc':sitedesc,     # might be None or commented
        'logo':logo,             # might be None or commented
        'unit':'nm',
        'resample': ("30", "min")
        }

In [ ]:
sn = SeismoNoise(displacement_RMS)
sn.plot(band="4.0-20.0")                          # timeseries
sn.plot(type="clockmaps")


In [ ]:
sn.plot(type="clockplots", band="4.0-20.0")

In [ ]:
sn.plot(type="dailyplots", band="4.0-20.0")